# 01 · Ground truth: COGERH lab results → `samples.csv`

COGERH publishes one trophic-state (IET) report per quarterly campaign. Each report has:
- **Table 3**: the lab results per reservoir (sample date, total N, total P, chlorophyll-a, cyanobacteria cells/mL, transparency);
- **Table 2**: stored volume (%) and trophic class.

This notebook reads every PDF in `data/raw/cogerh/`, checks the parser against four values copied from the Nov/2020 report, logs every row that could not be read, and writes `data/processed/samples.csv`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import RAW, PROCESSED, CYANO_THRESHOLD
from parse_cogerh import ParseLog, parse_report

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60)
pdfs = sorted((RAW / "cogerh").glob("*.pdf"))
[p.name for p in pdfs]

['iet_2019_08_ago.pdf', 'iet_2020_11_nov.pdf', 'iet_2021_08_ago.pdf']

## Parse every report

The PDFs have no ruled tables, so we read the text layer with `pdfplumber`. Every number is assigned to a column by its **horizontal position** under the column header. This matters because some cells are empty: reading the row left to right would shift the remaining values into the wrong columns. Rows we cannot read unambiguously are logged, never guessed. Values printed as `< x` (below the lab's detection limit) stay empty and keep the bound in the `censored` column.

In [2]:
log = ParseLog()
samples = pd.concat([parse_report(p, log) for p in pdfs], ignore_index=True)
print(samples.shape)
samples.head()

(421, 15)


,reservoir_raw,date,page,censored,n_total,p_total,chla,cyano,transparency,campaign,file,reservoir,volume_pct,trophic_class,qc_flag
0,Acarape do Meio,2019-08-06,12,,1.038,0.079,2.44,5868.0,1.0,2019-08,iet_2019_08_ago.pdf,acarape do meio,96.20,Mesotrófica,
1,Acaraú Mirim,2019-07-16,12,,0.688,0.042,2.90,19151.0,NaN,2019-08,iet_2019_08_ago.pdf,acarau mirim,99.68,Mesotrófica,
2,Adauto Bezerra,2019-08-14,12,,1.213,0.028,NaN,2120.0,NaN,2019-08,iet_2019_08_ago.pdf,adauto bezerra,0.00,Mesotrófica,
3,Amanary,2019-07-17,12,,1.438,0.083,16.99,13974.0,0.8,2019-08,iet_2019_08_ago.pdf,amanary,58.74,Eutrófica,
4,Angicos,2019-07-17,12,,0.938,0.049,7.08,13478.0,NaN,2019-08,iet_2019_08_ago.pdf,angicos,100.00,Mesotrófica,


## Parser test

These four rows were copied by hand from Table 3 of the Nov/2020 report. The parser must reproduce them exactly.

In [3]:
expected = pd.DataFrame([
    ("castanhao",               "2020-11-19",  9.64,  32_209, 2.40),
    ("oros",                    "2020-11-18", 43.41, 313_603, 0.80),
    ("banabuiu",                "2020-11-04", 15.40,  60_880, 1.50),
    ("santo antonio de russas", "2020-11-24", 30.13, 138_927, 0.50),
], columns=["reservoir", "date", "chla", "cyano", "transparency"])

nov = samples[samples["campaign"] == "2020-11"].set_index("reservoir")
for _, e in expected.iterrows():
    got = nov.loc[e["reservoir"]]
    assert got["date"] == pd.Timestamp(e["date"]), (e["reservoir"], got["date"])
    assert got["chla"] == e["chla"], (e["reservoir"], got["chla"])
    assert got["cyano"] == e["cyano"], (e["reservoir"], got["cyano"])
    assert got["transparency"] == e["transparency"], (e["reservoir"], got["transparency"])
    print("PASS", e["reservoir"])

PASS castanhao
PASS oros
PASS banabuiu
PASS santo antonio de russas


## What was skipped or flagged, and why

Every line the parser did not turn into a clean sample is in `parse_log.csv`. Expected entries: page notes ("Continua", "Fonte"), reservoirs that were dry, cancelled or covered by macrophytes, and empty cells. Anything else is a real problem to look at.

In [4]:
lg = log.frame()
lg["kind"] = lg["reason"].str.replace(r"(:.*| '.*)$", "", regex=True)
print(lg.groupby(["kind", "file"]).size().unstack(fill_value=0))
lg.to_csv(PROCESSED / "parse_log.csv", index=False)

file                                                          iet_2019_08_ago.pdf  iet_2020_11_nov.pdf  iet_2021_08_ago.pdf
kind                                                                                                                       
cyano value repeated in another reservoir (possible copy ...                    9                    6                    4
empty cells                                                                    92                    5                    1
no current-campaign cell                                                        4                    4                    4
no values (name only or note)                                                   4                    5                    5
status                                                                         38                   21                   27
unreadable chla value                                                           0                    0                    1
unreadab

In [5]:
lg[lg["kind"].isin(["unreadable cyano value", "unreadable chla value", "unreadable p_total value", "no valid date"])][["file", "page", "text", "reason"]]

,file,page,text,reason
149,iet_2020_11_nov.pdf,14,"Cocó 29/10/20 1,140 0,064 39,440 56,680","unreadable cyano value '56,680' -> NaN"
191,iet_2021_08_ago.pdf,14,"Broco 10/08/21 0,720 0,031 NR 20.175 1,70",unreadable chla value 'NR' -> NaN


In [6]:
lg[lg["kind"] == "status"]["reason"].str.replace("status: ", "").value_counts()

reason
Reservatório seco                      57
Reservatório coberto por macrófitas    15
Presença de macrófitas                  5
Presença de macrofitas                  5
Reservatório cancelado                  4
Name: count, dtype: int64

## Samples per campaign

Non-null counts per column, and the range of sample dates. Lab visits inside one campaign are spread over several weeks, so the real sample date (not the campaign month) is what we will match to satellite images.

In [7]:
samples.groupby("campaign").agg(
    rows=("reservoir", "size"), cyano=("cyano", "count"), chla=("chla", "count"),
    n_total=("n_total", "count"), p_total=("p_total", "count"),
    transparency=("transparency", "count"), volume=("volume_pct", "count"),
    first_date=("date", "min"), last_date=("date", "max"))

,rows,cyano,chla,n_total,p_total,transparency,volume,first_date,last_date
campaign,,,,,,,,,
2019-08,136,135,131,136,136,44,136,2019-06-27,2019-09-24
2020-11,144,142,141,144,144,142,144,2020-10-13,2020-12-22
2021-08,141,141,133,141,140,141,141,2021-07-01,2021-09-23


## Class balance for the target

`y = 1` when cyanobacteria exceed **20,000 cells/mL** (Portaria GM/MS 888/2021, Annex XX, art. 43: above this, weekly cyanotoxin testing is required).

In [8]:
lab = samples.dropna(subset=["cyano"]).copy()
lab["y"] = (lab["cyano"] > CYANO_THRESHOLD).astype(int)
lab.groupby("campaign")["y"].agg(n="size", positives="sum", share="mean").round(3)

,n,positives,share
campaign,,,
2019-08,135,88,0.652
2020-11,142,130,0.915
2021-08,141,131,0.929


## Possible copy errors

A few counts appear exactly twice (or three times) in different reservoirs of the same campaign. With six-digit numbers that is unlikely by chance, so it may be a copy error in the report. We keep the printed value, add a `qc_flag`, and later check that the results hold when these rows are removed.

In [9]:
samples.loc[samples["qc_flag"] != "", ["campaign", "reservoir_raw", "date", "cyano", "chla"]].sort_values(["campaign", "cyano"])

,campaign,reservoir_raw,date,cyano,chla
127,2019-08,Trapiá II,2019-07-16,3110.0,68.89
128,2019-08,Trapiá III,2019-07-10,3110.0,4.97
134,2019-08,Valério,2019-07-09,3110.0,1.00
24,2019-08,Cauhipe,2019-07-10,5443.0,1.56
25,2019-08,Caxitoré,2019-07-10,5443.0,1.56
38,2019-08,Escuridão,2019-07-04,8035.0,NaN
101,2019-08,Riacho da Serra,2019-07-03,8035.0,17.09
18,2019-08,Canoas,2019-07-02,109638.0,35.44
30,2019-08,Cupim,2019-07-23,109638.0,145.55
176,2020-11,Ema,2020-12-01,73622.0,31.87


## Reservoirs across campaigns

How many reservoirs appear in every campaign. A name that appears only once may be a spelling difference that breaks the join.

In [10]:
per_res = samples.groupby("reservoir")["campaign"].nunique()
print(per_res.value_counts().sort_index().rename("campaigns -> reservoirs"))
print(sorted(per_res[per_res == 1].index))

campaign
1      8
2     10
3    131
Name: campaigns -> reservoirs, dtype: int64
['barra velha', 'barragem do batalhao', 'carnaubal', 'madeiro', 'mons tabosa', 'muquem', 'potiretama', 'varzea do boi']


## Save

In [11]:
cols = ["reservoir", "date", "chla", "cyano", "n_total", "p_total", "transparency", "volume_pct", "campaign",
        "reservoir_raw", "trophic_class", "censored", "qc_flag", "file", "page"]
samples = samples.sort_values(["campaign", "reservoir"])[cols]
samples.to_csv(PROCESSED / "samples.csv", index=False, date_format="%Y-%m-%d")
print(samples.shape, "->", PROCESSED / "samples.csv")

(421, 15) -> C:\Users\Filipe\alerta-acude\data\processed\samples.csv
